# 01 — Cara Kerja LLM: Token, Next-Token Prediction, dan Sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/08-llm-basics/01_cara_kerja_llm.ipynb)

ChatGPT, Claude, Gemini, dan LLM lain terlihat seperti "bisa berpikir", padahal mekanisme dasarnya sederhana: **menebak token berikutnya**, lalu mengulanginya. Di modul 06 kamu sudah membuat mini-GPT tingkat karakter dari nol. Sekarang kita membedah LLM sungguhan yang sudah dilatih pada triliunan token.

Isi notebook:
1. **Tokenisasi**: teks → token → angka.
2. **Next-token prediction**: output model adalah distribusi probabilitas atas seluruh kosakata.
3. **Generasi autoregresif**: loop "tebak satu token, tambahkan, ulangi", ditulis manual lalu dibandingkan dengan `model.generate()`.
4. **Sampling**: temperature, top-k, dan top-p diimplementasikan sendiri, lalu divisualisasikan.
5. **Context window & KV cache**.
6. **Chat template**: bagaimana "percakapan" sebenarnya hanyalah teks dengan token khusus.

**Model:** [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (lisensi **Apache-2.0**), LLM kecil (±0,5 miliar parameter, unduhan ±1 GB) yang cukup paham bahasa Indonesia dan bisa berjalan di CPU. Sebagai pembanding, LLM frontier seperti Claude berukuran jauh lebih besar dan hanya bisa diakses lewat API (notebook 02).

> Notebook ini tidak butuh API key dan tidak wajib GPU. GPU (T4) membuat generasi lebih cepat.

## 0. Instalasi dan Memuat Model

`transformers` sudah tersedia di Colab; sel ini memastikan versinya cukup baru.

In [ ]:
%pip install -q "transformers>=4.56" accelerate pandas matplotlib

Memuat tokenizer dan model. `QUICK_RUN=1` (dipakai saat pengujian otomatis) hanya memperpendek jumlah token yang digenerate.

Model dimuat dalam `float32` agar hasilnya konsisten di CPU maupun GPU. `model.eval()` mematikan dropout karena kita hanya melakukan inference.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
N_TOKEN = 10 if QUICK_RUN else 30          # panjang teks yang digenerate di demo
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device).eval()

n_param = sum(p.numel() for p in model.parameters())
print(f"device: {device} | model: {MODEL_NAME} | {n_param / 1e6:.0f} juta parameter")
print(f"ukuran kosakata: {len(tokenizer):,} token | layer: {model.config.num_hidden_layers}")

Model dari Hugging Face biasanya membawa **setelan generasi bawaan** (misalnya sampling aktif, `temperature`, `repetition_penalty`). Agar setiap parameter yang kita pakai terlihat jelas, setelan itu diganti dengan setelan netral. Yang dipertahankan hanya ID token penanda akhir teks (EOS).

In [ ]:
print("Setelan bawaan:", model.generation_config.to_diff_dict())

eos = model.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])      # token yang menandai teks selesai
model.generation_config = GenerationConfig(eos_token_id=sorted(EOS_IDS), pad_token_id=tokenizer.pad_token_id)
print("EOS:", {i: tokenizer.decode([i]) for i in EOS_IDS})

## 1. Tokenisasi: Teks → Token → Angka

LLM tidak membaca huruf atau kata, melainkan **token**: potongan teks hasil tokenizer subword (BPE, lihat modul 06 notebook 01). Setiap token punya nomor (ID) di kosakata. Model hanya melihat deretan ID ini.

In [ ]:
kalimat = "Saya sedang belajar deep learning di Google Colab."
ids = tokenizer(kalimat)["input_ids"]
potongan = [tokenizer.decode([i]) for i in ids]

print(f"{len(ids)} token")
print("ID    :", ids)
print("Token :", potongan)

**Jumlah token menentukan biaya dan batas panjang input.** API LLM menagih per token, dan context window juga diukur dalam token. Tokenizer yang dilatih dominan pada teks bahasa Inggris biasanya memecah kalimat bahasa Indonesia menjadi lebih banyak token. Mari kita ukur dengan pasangan kalimat yang maknanya sama.

In [ ]:
pasangan = [
    ("Pengiriman barangnya cepat sekali dan kualitasnya memuaskan.",
     "The delivery was very fast and the quality is satisfying."),
    ("Bagaimana cara melatih jaringan saraf tiruan dengan data yang sedikit?",
     "How do I train an artificial neural network with little data?"),
    ("Pemerintah mengumumkan kebijakan baru untuk mengurangi kemacetan.",
     "The government announced a new policy to reduce traffic congestion."),
]

baris = []
for indo, inggris in pasangan:
    n_id, n_en = len(tokenizer(indo)["input_ids"]), len(tokenizer(inggris)["input_ids"])
    baris.append({"kalimat (id)": indo[:45] + "…", "token id": n_id, "token en": n_en,
                  "token/kata id": round(n_id / len(indo.split()), 2),
                  "token/kata en": round(n_en / len(inggris.split()), 2)})
pd.DataFrame(baris)

## 2. Next-Token Prediction

Untuk input berisi `n` token, model menghasilkan **logits** berukuran `[batch, n, ukuran_kosakata]`: satu vektor skor untuk **setiap posisi**. Skor di posisi terakhir adalah tebakan untuk token **berikutnya**. Softmax mengubah logits menjadi probabilitas (jumlahnya 1).

Selama pretraining, model dilatih meminimalkan cross-entropy antara distribusi ini dan token yang sebenarnya muncul, persis seperti klasifikasi dengan ratusan ribu kelas.

In [ ]:
prompt = "Ibu kota Indonesia adalah"
enc = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**enc).logits

print("shape input_ids:", tuple(enc["input_ids"].shape))
print("shape logits   :", tuple(logits.shape), "→ [batch, posisi, kosakata]")

probs = F.softmax(logits[0, -1], dim=-1)          # distribusi token berikutnya
top = torch.topk(probs, 10)
pd.DataFrame({"token berikutnya": [repr(tokenizer.decode([i])) for i in top.indices.tolist()],
              "probabilitas": top.values.cpu().numpy().round(4)})

Perhatikan bahwa model tidak "menyimpan jawaban", melainkan memberi probabilitas ke **semua** token. Token yang masuk akal mendapat probabilitas tinggi, tetapi token lain tetap punya peluang kecil. Cara memilih token dari distribusi inilah yang membedakan jawaban yang kaku dan jawaban yang kreatif (bagian 4).

## 3. Generasi Autoregresif: Loop Manual

Teks panjang dihasilkan dengan loop:
1. Hitung distribusi token berikutnya.
2. Pilih satu token (di sini: **greedy**, yaitu yang probabilitasnya paling tinggi).
3. Tambahkan token itu ke input, lalu ulangi.

Setiap token baru bergantung pada semua token sebelumnya, termasuk token buatan model sendiri. Itu sebabnya satu kesalahan di awal bisa "terbawa" sampai akhir jawaban.

In [ ]:
@torch.no_grad()
def generate_greedy(prompt, n_token=N_TOKEN):
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    for _ in range(n_token):
        logits = model(input_ids=ids).logits[0, -1]     # skor token berikutnya
        next_id = logits.argmax()                        # greedy: ambil yang tertinggi
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
        if next_id.item() in EOS_IDS:                    # model menandai teks selesai
            break
    return tokenizer.decode(ids[0], skip_special_tokens=True)


prompt = "Langkah-langkah melatih model deep learning adalah"
hasil_manual = generate_greedy(prompt)
print(hasil_manual)

`model.generate()` dari Hugging Face melakukan hal yang sama (dengan banyak optimasi). Dengan `do_sample=False` (greedy), hasilnya seharusnya identik dengan loop manual kita.

In [ ]:
enc = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=N_TOKEN, do_sample=False)
hasil_hf = tokenizer.decode(out[0], skip_special_tokens=True)

print(hasil_hf)
print("\nSama dengan loop manual:", hasil_hf == hasil_manual)

## 4. Sampling: Temperature, Top-k, dan Top-p

Greedy selalu memberi jawaban yang sama dan cenderung monoton (bahkan bisa berulang-ulang). **Sampling** memilih token secara acak sesuai probabilitasnya. Tiga parameter populer untuk mengaturnya:

| Parameter | Cara kerja | Efek |
|---|---|---|
| **temperature** `T` | logits dibagi `T` sebelum softmax | `T < 1`: distribusi makin tajam (lebih yakin, lebih konsisten). `T > 1`: makin datar (lebih beragam, lebih ngawur). `T → 0` sama dengan greedy. |
| **top-k** | hanya `k` token teratas yang boleh dipilih | Membuang "ekor" token yang sangat tidak mungkin. |
| **top-p** (nucleus) | ambil token teratas sampai probabilitas kumulatifnya ≥ `p` | Seperti top-k, tetapi jumlah token menyesuaikan tingkat keyakinan model. |

Kita implementasikan sendiri agar tidak ada yang "ajaib".

In [ ]:
def atur_distribusi(logits, temperature=1.0, top_k=None, top_p=None):
    """Ubah logits menjadi probabilitas setelah temperature, top-k, dan top-p."""
    logits = logits / temperature
    if top_k is not None:
        batas = torch.topk(logits, top_k).values[-1]
        logits = logits.masked_fill(logits < batas, float("-inf"))   # exp(-inf) = 0
    probs = F.softmax(logits, dim=-1)
    if top_p is not None:
        urut, idx = torch.sort(probs, descending=True)
        kumulatif_sebelum = urut.cumsum(0) - urut                    # massa probabilitas sebelum token ini
        urut = urut.masked_fill(kumulatif_sebelum >= top_p, 0.0)      # buang token setelah kumulatif mencapai p
        probs = torch.zeros_like(probs).scatter(0, idx, urut)
        probs = probs / probs.sum()                                  # normalisasi ulang
    return probs


prompt_sampling = "Hari ini saya ingin makan"
with torch.no_grad():
    logits_akhir = model(**tokenizer(prompt_sampling, return_tensors="pt").to(device)).logits[0, -1]

# Token teratas menurut distribusi asli (T=1) dipakai sebagai sumbu x semua grafik
urutan = torch.topk(logits_akhir, 12).indices
label = [repr(tokenizer.decode([i])) for i in urutan.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, t in zip(axes, [0.5, 1.0, 1.5]):
    p = atur_distribusi(logits_akhir, temperature=t)[urutan].cpu().numpy()
    ax.bar(range(len(p)), p)
    ax.set_xticks(range(len(p)), label, rotation=70, fontsize=8)
    ax.set_title(f"temperature = {t}")
axes[0].set_ylabel("probabilitas")
fig.suptitle(f"Distribusi token setelah: '{prompt_sampling}'")
plt.tight_layout()
plt.show()

Angka ringkas: **entropi** (makin tinggi = makin tidak pasti), probabilitas token teratas, dan jumlah token yang lolos saringan top-p 0,9 untuk tiap temperature.

In [ ]:
baris = []
for t in [0.2, 0.5, 1.0, 1.5, 2.0]:
    p = atur_distribusi(logits_akhir, temperature=t)
    entropi = -(p * torch.log(p.clamp_min(1e-12))).sum().item()
    lolos_top_p = (atur_distribusi(logits_akhir, temperature=t, top_p=0.9) > 0).sum().item()
    baris.append({"temperature": t, "entropi": round(entropi, 2), "prob token teratas": round(p.max().item(), 3),
                  "token lolos top-p 0.9": lolos_top_p})
pd.DataFrame(baris)

Sekarang generasi dengan sampling. Fungsi ini sama dengan loop greedy, hanya `argmax` diganti `torch.multinomial` (mengambil sampel sesuai probabilitas). Setiap setelan dijalankan 3 kali untuk melihat keragamannya.

In [ ]:
@torch.no_grad()
def generate_sampling(prompt, n_token=N_TOKEN, seed=None, **setelan):
    if seed is not None:
        torch.manual_seed(seed)
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    for _ in range(n_token):
        probs = atur_distribusi(model(input_ids=ids).logits[0, -1], **setelan)
        next_id = torch.multinomial(probs, num_samples=1)          # ambil 1 token secara acak
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
        if next_id.item() in EOS_IDS:
            break
    return tokenizer.decode(ids[0], skip_special_tokens=True)


for setelan in [dict(temperature=0.2), dict(temperature=0.8, top_p=0.9), dict(temperature=1.8)]:
    print(f"=== {setelan} ===")
    for seed in range(3):
        print(f"  [{seed}]", generate_sampling(prompt_sampling, seed=seed, **setelan).replace("\n", " "))

Biasanya terlihat:
- `temperature=0.2`: ketiga hasil hampir sama (mendekati greedy).
- `temperature=0.8, top_p=0.9`: beragam tetapi masih masuk akal. Ini setelan umum untuk chat.
- `temperature=1.8`: sangat beragam, sering tidak nyambung karena token dari "ekor" distribusi ikut terpilih.

Di `model.generate()` setelan yang sama cukup ditulis sebagai argumen:

In [ ]:
torch.manual_seed(0)
enc = tokenizer(prompt_sampling, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=N_TOKEN, do_sample=True, temperature=0.8, top_p=0.9,
                         top_k=0)   # top_k=0: top-k tidak dipakai, sama dengan fungsi manual kita
print(tokenizer.decode(out[0], skip_special_tokens=True))

> **Catatan untuk API:** banyak API LLM menerima `temperature`/`top_p`. Namun model Claude terbaru (termasuk `claude-opus-5-5` yang dipakai di notebook 02) mengatur sampling sendiri, dan SDK `anthropic` versi 1.x tidak lagi menyediakan parameter tersebut. Perilaku model di sana diarahkan lewat **prompt** dan parameter **effort**. Karena itu sampling dipelajari di sini, dengan model lokal yang bisa kita bedah.

## 5. Context Window dan KV Cache

**Context window** adalah jumlah token maksimum (prompt + jawaban) yang bisa diproses model sekaligus. LLM **tidak punya ingatan** di luar context window: setiap permintaan hanya "melihat" token yang ada di input saat itu. Aplikasi chat terlihat "ingat" karena riwayat percakapan dikirim ulang setiap giliran (dipraktikkan di notebook 02).

Mengapa context window terbatas? Self-attention membandingkan setiap token dengan semua token sebelumnya, sehingga komputasinya tumbuh **kuadratik** terhadap panjang input.

In [ ]:
print(f"Context window {MODEL_NAME}: {model.config.max_position_embeddings:,} token")

for n in [1_000, 10_000, 100_000]:
    print(f"{n:>7,} token → {n * n / 1e6:>10,.0f} juta skor attention per head per layer")

Loop manual di bagian 3 menghitung ulang **semua** posisi setiap kali satu token ditambahkan. Padahal representasi (key & value) token lama tidak berubah. **KV cache** menyimpannya sehingga tiap langkah hanya memproses 1 token baru. `model.generate()` memakai KV cache secara default. Bandingkan kecepatannya:

In [ ]:
enc = tokenizer("Jelaskan sejarah singkat komputer:", return_tensors="pt").to(device)
n_baru = 20 if QUICK_RUN else 60

for pakai_cache in [False, True]:
    mulai = time.perf_counter()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=n_baru, min_new_tokens=n_baru, do_sample=False, use_cache=pakai_cache)
    durasi = time.perf_counter() - mulai
    print(f"use_cache={pakai_cache!s:5} → {durasi:.2f} detik untuk {n_baru} token ({n_baru / durasi:.1f} token/detik)")

KV cache menukar **memori** dengan kecepatan. Untuk context yang sangat panjang, ukuran cache ini (dan memori GPU) menjadi salah satu biaya utama menjalankan LLM. Hal ini juga menjadi alasan API menagih token input, dan memberi diskon untuk *prompt caching* (awalan prompt yang sama dipakai ulang).

## 6. Chat Template: Percakapan Hanyalah Teks

Model **base** hanya dilatih melanjutkan teks. Model **instruct/chat** (seperti `Qwen2.5-0.5B-Instruct`) dilatih lanjut agar mengikuti instruksi dalam format percakapan. Format itu tetap berupa **teks biasa** dengan token khusus yang menandai peran (`system`, `user`, `assistant`). `apply_chat_template` mengubah daftar pesan menjadi teks tersebut.

In [ ]:
pesan = [
    {"role": "system", "content": "Kamu asisten yang menjawab singkat dalam bahasa Indonesia."},
    {"role": "user", "content": "Apa itu overfitting? Jawab dalam 2 kalimat."},
]
teks_prompt = tokenizer.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
print(teks_prompt)

`add_generation_prompt=True` menambahkan penanda awal giliran `assistant`, sehingga token berikutnya yang paling mungkin adalah jawaban. Kita generate lalu hanya men-decode token **baru** (setelah prompt).

In [ ]:
enc = tokenizer(teks_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=40 if QUICK_RUN else 120, do_sample=False)

jawaban = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
print(jawaban)

Hal yang sama terjadi saat kamu memanggil API seperti Claude: daftar `messages` yang kamu kirim diubah server menjadi teks berformat seperti di atas, lalu model melakukan next-token prediction sampai menghasilkan token "selesai". Bedanya, model di balik API jauh lebih besar dan dilatih lebih lanjut agar membantu, jujur, dan aman.

Model 0,5 miliar parameter ini cepat dan murah, tetapi mudah salah fakta dan kurang mampu bernalar. Perbedaan kualitas ini akan terlihat jelas saat kamu mencoba pertanyaan yang sama di notebook 02.

## Ringkasan

- LLM bekerja pada **token**. Jumlah token menentukan biaya dan batas panjang input, dan teks bahasa Indonesia sering butuh lebih banyak token.
- Output model = **distribusi probabilitas** token berikutnya. Teks panjang dibuat **autoregresif**: tebak, tambahkan, ulangi.
- **Greedy** deterministik; **sampling** dengan temperature/top-k/top-p mengatur keseimbangan konsisten ↔ beragam.
- **Context window** membatasi apa yang "diingat" model; attention tumbuh kuadratik dan **KV cache** mempercepat generasi.
- **Chat template** mengubah percakapan menjadi teks berpenanda peran; model instruct dilatih mengikuti format ini.

## Latihan

1. **Temperature ekstrem.** Jalankan `generate_sampling` dengan `temperature=0.01` dan bandingkan dengan `generate_greedy`. Lalu coba `temperature=5`. Jelaskan secara matematis (lihat rumus softmax `exp(z/T) / Σ exp(z/T)`) mengapa `T → 0` sama dengan argmax dan `T → ∞` mendekati distribusi seragam.
2. **Top-k vs top-p.** Cari dua prompt: satu yang jawabannya hampir pasti (misalnya `"Satu, dua, tiga, empat,"`) dan satu yang sangat terbuka (misalnya `"Nama kucing saya adalah"`). Untuk keduanya, hitung berapa token yang lolos top-p 0,9 dan bandingkan dengan top-k tetap `k=10`. Mengapa top-p dianggap lebih adaptif?
3. **Model kecil vs lebih besar.** Unduh 20 kalimat NusaX-Senti (lihat modul 06) dan minta model mengklasifikasikan sentimennya dengan chat template (jawab satu kata: positif/negatif/netral). Hitung akurasi `Qwen2.5-0.5B-Instruct`, lalu ulangi dengan `Qwen/Qwen2.5-1.5B-Instruct` (juga Apache-2.0, sebaiknya dengan GPU). Catat juga berapa jawaban yang formatnya tidak sesuai instruksi.